# Building Generative AI Agents with the Agent Development Kit (ADK)

Adopted from https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/asl_genai/notebooks/building_agents/labs/building_agent_with_adk.ipynb

## Overview

This notebook guides you through building a conversational AI agent, using the Agent Development Kit (ADK). We'll start with a simple agent that looks up weather information and progressively add more sophisticated features like sub-agent delegation for handling different types of user requests and session state for memory and personalization. <br>
Throughout this tutorial, you'll also learn how to use the ADK Developer UI to inspect and debug your agents.

## Learning Goals

By the end of this notebook, you will understand how to:
* Create and configure **Agents**, using **Tools**, and **Sub-Agents**.
* Use the ADK **Runner** and **SessionService** to manage and execute agent interactions.
* Set up and use the **ADK Developer UI** for inspecting agent behavior, including tool calls and event flows.
* Implement **Session State** to provide agents with memory, enabling context-aware and personalized conversations.

## Install Packages

In [ ]:
import asyncio
import importlib
import os
import warnings

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm  # For multi-model support
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext
from google.genai import types  # For creating message Content/Parts
from IPython.display import HTML, Markdown, display

# Ignore all warnings
warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.ERROR)

# Ignore opentelemetry errors
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

In [ ]:
LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [ ]:
MODEL = "gemini-2.5-flash"

In [ ]:
from adk_agents.agent1_mlb_get_game import agent

importlib.reload(agent)  # Force reload
print(agent.get_games_on_date("2022-11-16"))

### Setup Runner and Session Service

To manage conversations and execute the agent, we need two more components:

* `SessionService`: Responsible for managing conversation history and state for different users and sessions. The `InMemorySessionService` is a simple implementation that stores everything in memory, suitable for testing and simple applications. It keeps track of the messages exchanged. We'll explore state persistence more in Step 4\.  
* `Runner`: The engine that orchestrates the interaction flow. It takes user input, routes it to the appropriate agent, manages calls to the LLM and tools based on the agent's logic, handles session updates via the `SessionService`, and yields events representing the progress of the interaction.

Let's define some constants first.

In [ ]:
APP_NAME = "mlb_chatbot_app"
USER_ID = "user_1"
SESSION_ID = "session_001"  # Using a fixed ID for simplicity

#### Session

In [ ]:
session_service = InMemorySessionService()

# Create the specific session where the conversation will happen
session = await session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
)

#### Runner
Key Concept: Runner orchestrates the agent execution loop.

In [ ]:
runner = Runner(
    agent=agent.root_agent,  # The agent we want to run
    app_name=APP_NAME,  # Associates runs with our app
    session_service=session_service,  # Uses our session manager
)

### Interact with the Agent

We need a way to send messages to our agent and receive its responses. Since LLM calls and tool executions can take time, ADK's `Runner` operates asynchronously.

We'll define an `async` helper function (`call_agent_async`) that:

1. Takes a user query string.  
2. Packages it into the ADK `Content` format.  
3. Calls `runner.run_async`, providing the user/session context and the new message.  
4. Iterates through the **Events** yielded by the runner. Events represent steps in the agent's execution (e.g., tool call requested, tool result received, intermediate LLM thought, final response).  
5. Identifies and prints the **final response** event using `event.is_final_response()`.

**Why `async`?** Interactions with LLMs and potentially tools (like external APIs) are I/O-bound operations. Using `asyncio` allows the program to handle these operations efficiently without blocking execution.

In [ ]:
async def call_agent_async(query: str, runner, user_id, session_id):
    """Sends a query to the agent and prints the final response."""
    print(f"\n>>> User Query: {query}")

    content = types.Content(role="user", parts=[types.Part(text=query)])

    final_response_text = "Agent did not produce a final response."  # Default

    # Key Concept: run_async executes the agent logic and yields Events.
    # We iterate through events to find the final answer.
    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=content
    ):
        # You can uncomment the line below to see *all* events during execution
        # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

        # Key Concept: is_final_response() marks the concluding message for the turn.
        if event.is_final_response():
            if event.content and event.content.parts:
                # Assuming text response in the first part
                final_response_text = event.content.parts[0].text
            elif (
                event.actions and event.actions.escalate
            ):  # Handle potential errors/escalations
                final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
            # Add more checks here if needed (e.g., specific error codes)
            break  # Stop processing events once the final response is found

    print(f"<<< Agent Response: {final_response_text}")

### Run the Conversation

Finally, let's test our setup by sending a few queries to the agent. We wrap our `async` calls in a main `async` function and run it using `await`.

Watch the output:

* See the user queries.  
* Notice the `--- Tool: get_weather called... ---` logs when the agent uses the tool.  
* Observe the agent's final responses, including how it handles the case where weather data isn't available (for Paris).

In [ ]:
await call_agent_async(
    "What were the games on 2022-11-16?",
    runner=runner,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

You can launch the ADK Dev UI to interact with your agents using the `adk web` command. 

Execute the cell below and open the printed URL. (It may take a few seconds)

**Note**: You can also run `adk web` via the terminal. If you do so, ensure your virtual environment is activated first.

```
# on asl-ml-immersion directory
source ./asl_genai/.venv/bin/activate
adk web ./asl_genai/notebooks/building_agents/solutions/adk_agents
```

In [ ]:
# On Cloud Workstations
!adk web adk_agents --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note:** if you are using Agent Platform Workbench, remove the comment out and run the cell below.

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web adk_agents --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

In [ ]:
!cp ./adk_agents/agent1_weather_lookup/tools.py ./adk_agents/agent2_sub_agent/

Next, let's create the Python functions that will serve as tools for our new greeting and farewell specialist agents. Remember, clear and descriptive docstrings are vital for the LLMs of the agents that will use them.

In [ ]:
%%writefile -a ./adk_agents/agent2_sub_agent/tools.py

def say_hello(name: str = "there") -> str:
    """Provides a simple greeting, optionally addressing the user by name.

    Args:
        name (str, optional): The name of the person to greet. Defaults to "there".

    Returns:
        str: A friendly greeting message.
    """
    if name is None or name.strip() == "":
        name = "there"
    print(f"--- Tool: say_hello called with name: {name} ---")
    return f"Hello, {name}!"

def say_goodbye() -> str:
    """Provides a simple farewell message to conclude the conversation."""
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

In [ ]:
from adk_agents.agent2_sub_agent import tools

print(tools.say_hello("Alice"))
print(tools.say_goodbye())

### Define the Sub-Agents (Greeting & Farewell)

Now, we'll create the `Agent` instances for our specialist sub-agents. Notice their highly focused `instruction` prompts and, critically, their clear `description` fields. The `description` is the primary piece of information the *root agent's* LLM uses to determine *when* to delegate a task to one of these sub-agents.

**Best Practice:**
* A sub-agent's `description` field should accurately and concisely summarize its specific capability. This is crucial for effective automatic delegation.
* A sub-agent's `instruction` field should be tailored to its limited scope, telling it exactly what to do and, importantly, *what not* to do (e.g., "Your *only* task is to...").

In [ ]:
%%writefile ./adk_agents/agent2_sub_agent/agent.py
from google.adk.agents import Agent
MODEL = "gemini-2.5-flash"

from .tools import get_weather, say_hello, say_goodbye

# --- Greeting Agent ---
greeting_agent = Agent(
    model=MODEL,
    name="greeting_agent",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. "
                "Use the 'say_hello' tool to generate the greeting. "
                "If the user provides their name, make sure to pass it to the tool. "
                "Do not engage in any other conversation or tasks.",
    description="Handles simple greetings and hellos using the 'say_hello' tool.", # Crucial for delegation
    tools=[say_hello],
)

In [ ]:
%%writefile -a ./adk_agents/agent2_sub_agent/agent.py

# --- Farewell Agent ---
farewell_agent = Agent(
    model=MODEL,
    name="farewell_agent",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. "
                "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                "Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", # Crucial for delegation
    tools=[say_goodbye],
)

### Define the Root Agent (`weather_agent_v2`) with Sub-Agents

Now, we'll define `weather_agent_v2` to act as our root agent. The key changes from `weather_agent_v1` are:

* Adding the `sub_agents` parameter: We pass a list containing the `greeting_agent` and `farewell_agent` instances we just created.
* Updating the `instruction`: We explicitly tell the root agent *about* its sub-agents and *when* it should delegate tasks to them.

**Key Concept: Automatic Delegation (Auto Flow)**
By providing the `sub_agents` list to an agent, ADK enables automatic delegation. When the root agent (in this case, `weather_agent_v2`) receives a user query, its LLM considers not only its own instructions and tools but also the `description` of each sub-agent. If the LLM determines that a query aligns better with a sub-agent's described capability (e.g., the `greeting_agent`'s description: "Handles simple greetings..."), it will automatically generate a special internal action to *transfer control* to that sub-agent for that conversational turn. The chosen sub-agent then processes the query using its own LLM, instructions, and tools.

**Best Practice:** Ensure the root agent's instructions clearly guide its delegation decisions. Mention the sub-agents (often by `name`) and describe the conditions under which delegation to each should occur.

In [ ]:
%%writefile -a ./adk_agents/agent2_sub_agent/agent.py

root_agent = Agent(
    name="weather_agent_v2", # Give it a new version name
    model=MODEL,
    description="The main coordinator agent. Handles weather requests and delegates greetings/farewells to specialists.",
    instruction="You are the main Weather Agent coordinating a team. Your primary responsibility is to provide weather information. "
                "Use the 'get_weather' tool ONLY for specific weather requests (e.g., 'weather in London'). "
                "You have specialized sub-agents: "
                "1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. "
                "2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. "
                "Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. If it's a farewell, delegate to 'farewell_agent'. "
                "If it's a weather request, handle it yourself using 'get_weather'. "
                "For anything else, respond appropriately or state you cannot handle it.",
    tools=[get_weather],
    sub_agents=[greeting_agent, farewell_agent]
)

### Interact with the Agent Team

Now that we've defined our root agent (`weather_agent_v2`) with its specialized sub-agents, let's test the delegation mechanism.

We will set up a new `Runner` and `SessionService` for this agent team to keep the interactions distinct. Then, we'll send a series of queries to demonstrate the delegation.

**Expected Flow:**

1.  The "Hello there!" query will be sent to the `runner_agent_team`.
2.  The root agent (`weather_agent_v2`) will receive it. Based on its instructions and the `greeting_agent`'s `description`, it should delegate the task.
3.  The `greeting_agent` will then handle the query, likely calling its `say_hello` tool to generate the response.
4.  The "What is the weather in New York?" query should *not* be delegated. The root agent will handle it directly using its `get_weather` tool.
5.  The "Thanks, bye!" query should be delegated to the `farewell_agent`, which will use its `say_goodbye` tool.

Observe the tool execution logs (`--- Tool: ... called ---`) to see which agent's tool is being invoked.

In [ ]:
from adk_agents.agent2_sub_agent import agent as agent2

importlib.reload(agent2)  # Force reload

In [ ]:
APP_NAME = "weather_tutorial_agent_team"
USER_ID = "user_1_agent_team"
SESSION_ID = "session_001_agent_team"

session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
)

runner_agent_team = Runner(  # Or use InMemoryRunner
    agent=agent2.root_agent, app_name=APP_NAME, session_service=session_service
)

In [ ]:
await call_agent_async(
    query="Hello there!",
    runner=runner_agent_team,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

In [ ]:
await call_agent_async(
    query="What is the weather in New York?",
    runner=runner_agent_team,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

In [ ]:
await call_agent_async(
    query="Thanks, bye!",
    runner=runner_agent_team,
    user_id=USER_ID,
    session_id=SESSION_ID,
)

### Inspecting the Agent Team with the ADK Developer UI

You can use the ADK Developer UI again to observe the agent team's behavior.

1.  Run `adk web` and open the Web UI.
2.  Select `agent2_sub_agent` (which contains our `weather_agent_v2` as the root agent) from the agent dropdown.
3.  Send the same queries: "Hello there!", "What is the weather in New York?", and "Thanks, bye!".
4.  In the "Events" panel, look for the transfer to agent action events. These indicate that the root agent has delegated the task to a sub-agent. You can then see the sub-agent's subsequent actions. This is a powerful way to visualize and debug the delegation flow.
5.  When finished, interrupt the kernel in your notebook and stop the Developer UI.

In [ ]:
# On Cloud Workstations
!adk web adk_agents --allow_origins "regex:https://.*\.cloudworkstations\.dev"

**Note:** if you are using Agent Platform Workbench, remove the comment out and run the cell below.

In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web adk_agents --url_prefix /proxy/8000 --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"